# 00 · Organização do Ambiente
### V-Energy Grid 360 — Arquitetura Medalhão

Objetivos deste notebook:
1. Definir os parâmetros do ambiente (catalog, schemas, caminho da Landing)
2. Criar o `catalog`, os `schemas` e o **Volume** de Landing no Unity Catalog
3. Validar que os arquivos de origem foram enviados para a Landing
4. Inspecionar o formato de cada arquivo (separador, encoding, cabeçalho) antes de escrever a ingestão

**Cada integrante roda este notebook na própria conta do Databricks.** Tudo é `IF NOT EXISTS`, então pode ser reexecutado sem problema.

## 1. Parâmetros do ambiente

In [0]:
# Padronização de nomes: <catalog>.<camada>.<tabela>, tudo em snake_case
catalog = "v_energy"
bronze_schema_name = "bronze"
silver_schema_name = "silver"
gold_schema_name = "gold"

bronze_schema = f"{catalog}.{bronze_schema_name}"
silver_schema = f"{catalog}.{silver_schema_name}"
gold_schema = f"{catalog}.{gold_schema_name}"

landing_path = f"/Volumes/{catalog}/{bronze_schema_name}/landing"

print(f"catalog: {catalog}")
print(f"bronze_schema: {bronze_schema}")
print(f"silver_schema: {silver_schema}")
print(f"gold_schema: {gold_schema}")
print(f"landing_path: {landing_path}")

## 2. Criação do Catalog, Schemas e Volume

In [0]:
spark.sql(f"CREATE CATALOG IF NOT EXISTS {catalog}")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {bronze_schema}")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {silver_schema}")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {gold_schema}")
spark.sql(f"CREATE VOLUME IF NOT EXISTS {bronze_schema}.landing")

print("Catalog, schemas e volume prontos.")

## 3. Upload dos arquivos de origem

- `unidades_consumidoras.csv`
- `faturas.csv`
- `medidores.csv`
- `ocorrencias_rede.csv`
- `atendimentos.json`
- `geracao_distribuida.csv`
- `inspecoes.xlsx`


## 4. Validação da Landing Zone

In [0]:
expected_files = [
    "unidades_consumidoras.csv",
    "faturas.csv",
    "medidores.csv",
    "ocorrencias_rede.csv",
    "atendimentos.json",
    "geracao_distribuida.csv",
    "inspecoes.xlsx",
]
expected_leituras = 5

try:
    existing = {f.name for f in dbutils.fs.ls(landing_path)}
except Exception as e:
    existing = set()
    print(f"[ALERTA] Não foi possível listar '{landing_path}'. Faça o upload antes de continuar.\n{e}")

missing = [f for f in expected_files if f not in existing]

try:
    leituras = [f for f in dbutils.fs.ls(f"{landing_path}/leituras") if not f.name.endswith("/")]
except Exception:
    leituras = []

if missing:
    print("[PENDENTE] Arquivos ainda não encontrados na landing zone:")
    for f in missing:
        print(f"  - {f}")
if len(leituras) != expected_leituras:
    print(f"[PENDENTE] Esperados {expected_leituras} arquivos em leituras/, encontrados {len(leituras)}")

if not missing and len(leituras) == expected_leituras:
    print("[OK] Todos os arquivos esperados estão na landing zone:")

for f in dbutils.fs.ls(landing_path):
    print(f"  - {f.name} ({f.size/1e6:.1f} MB)")
for f in leituras:
    print(f"  - leituras/{f.name} ({f.size/1e6:.1f} MB)")

## 5. Inspeção do formato dos arquivos

Mostra o início de cada CSV/JSON para descobrir **separador, encoding, formato de datas/decimais e estrutura do JSON**. Essas informações definem como o notebook `01_ingestao_bronze` vai ler cada arquivo.

In [0]:
import os

arquivos = [os.path.join(landing_path, f) for f in sorted(os.listdir(landing_path))]
arquivos += [os.path.join(landing_path, "leituras", f) for f in sorted(os.listdir(f"{landing_path}/leituras"))]

for caminho in arquivos:
    if not caminho.endswith((".csv", ".json")):
        continue
    with open(caminho, "rb") as f:
        inicio = f.read(500)
    print("=" * 90)
    print(caminho.replace(landing_path + "/", ""))
    print("-" * 90)
    try:
        print(inicio.decode("utf-8"))
    except UnicodeDecodeError:
        print("[NÃO É UTF-8 — mostrando como latin-1]")
        print(inicio.decode("latin-1"))

### Excel de inspeções

Mostra as 6 primeiras linhas de cada aba **sem assumir cabeçalho**, para ver se o cabeçalho está mesmo na primeira linha (planilhas preenchidas à mão costumam ter título, linhas vazias ou células mescladas).

In [0]:
%pip install openpyxl -q

In [0]:
import pandas as pd

landing_path = "/Volumes/v_energy/bronze/landing"  # redefinido porque o %pip reinicia o Python

abas = pd.read_excel(f"{landing_path}/inspecoes.xlsx", sheet_name=None, header=None, nrows=6, dtype=str)
for nome_aba, df in abas.items():
    print(f"=== Aba: {nome_aba} ===")
    display(df)